### Baseline interpretation

The majority-class baseline predicts every transaction as legitimate.
Because fraudulent transactions represent only a very small proportion
of the dataset, this produces extremely high overall accuracy despite
detecting no fraudulent transactions.

This demonstrates why accuracy is unsuitable as the primary evaluation
metric for FinGuard. Subsequent models will instead be evaluated using
fraud-class precision and recall, F1-score, PR-AUC and the associated
false-positive trade-off.

In [1]:
import pandas as pd
import numpy as np

DATA_PATH = "../data/raw/paysim.csv"

df = pd.read_csv(DATA_PATH)

print(df.shape)

(6362620, 11)


In [4]:
# origin balance error
df["orig_balance_error"] = (
    df["oldbalanceOrg"]
    - df["amount"]
    - df["newbalanceOrig"]
).abs()

df["amount_to_orig_balance"] = (
    df["amount"] /
    (df["oldbalanceOrg"] + 1)
)

In [3]:
# destination balance error
df["dest_balance_error"] = (
    df["oldbalanceDest"]
    + df["amount"]
    - df["newbalanceDest"]
).abs()

df["amount_to_dest_balance"] = (
    df["amount"] /
    (df["oldbalanceDest"] + 1)
)

In [5]:
df["hour"] = (df["step"] - 1) % 24
df["day"] = (df["step"] - 1) // 24

In [7]:
# what the modle is allowed to see
drop_columns = [
    "nameOrig",
    "nameDest",
    "isFlaggedFraud"
]

df_model = df.drop(columns=drop_columns)
df_model.columns.tolist()

['step',
 'type',
 'amount',
 'oldbalanceOrg',
 'newbalanceOrig',
 'oldbalanceDest',
 'newbalanceDest',
 'isFraud',
 'orig_balance_error',
 'dest_balance_error',
 'amount_to_dest_balance',
 'amount_to_orig_balance',
 'hour',
 'day']

In [8]:
df.groupby("isFraud")[
    [
        "orig_balance_error",
        "dest_balance_error",
        "amount_to_orig_balance",
        "amount_to_dest_balance"
    ]
].median()

,orig_balance_error,dest_balance_error,amount_to_orig_balance,amount_to_dest_balance
isFraud,,,,
0,69049.31,5123.10,6.511566,0.915134
1,0.00,9511.69,0.999998,116419.580000


In [9]:
df.groupby("isFraud")[
    [
        "orig_balance_error",
        "dest_balance_error"
    ]
].mean()

,orig_balance_error,dest_balance_error
isFraud,,
0,201338.558304,92756.964361
1,10692.325265,745138.585637


In [10]:
df[
    [
        "amount",
        "oldbalanceOrg",
        "newbalanceOrig",
        "oldbalanceDest",
        "newbalanceDest",
        "orig_balance_error",
        "dest_balance_error"
    ]
].describe()

,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,orig_balance_error,dest_balance_error
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06,2.010925e+05,9.359907e+04
std,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06,6.066505e+05,4.350570e+05
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.954230e+03,0.000000e+00
50%,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,6.867726e+04,5.123620e+03
75%,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,2.496411e+05,4.342133e+04
max,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,9.244552e+07,7.588573e+07


In [11]:
df = pd.get_dummies(
    df,
    columns=["type"],
    prefix="type",
    dtype=int
)

In [12]:
df.columns.tolist()

['step',
 'amount',
 'nameOrig',
 'oldbalanceOrg',
 'newbalanceOrig',
 'nameDest',
 'oldbalanceDest',
 'newbalanceDest',
 'isFraud',
 'isFlaggedFraud',
 'orig_balance_error',
 'dest_balance_error',
 'amount_to_dest_balance',
 'amount_to_orig_balance',
 'hour',
 'day',
 'type_CASH_IN',
 'type_CASH_OUT',
 'type_DEBIT',
 'type_PAYMENT',
 'type_TRANSFER']

In [13]:
drop_columns = [
    "nameOrig",
    "nameDest",
    "isFlaggedFraud"
]

df_model = df.drop(columns=drop_columns)

In [14]:
split_step = int(df_model["step"].max() * 0.8)

split_step

594

In [15]:
train_df = df_model[df_model["step"] <= split_step].copy()
test_df = df_model[df_model["step"] > split_step].copy()

print("Training:", train_df.shape)
print("Testing:", test_df.shape)

print("\nTraining steps:",
      train_df["step"].min(),
      "to",
      train_df["step"].max())

print("Testing steps:",
      test_df["step"].min(),
      "to",
      test_df["step"].max())

Training: (6239040, 18)
Testing: (123580, 18)

Training steps: 1 to 594
Testing steps: 595 to 743


In [16]:
print("TRAIN")
print(train_df["isFraud"].value_counts())
print(train_df["isFraud"].value_counts(normalize=True) * 100)

print("\nTEST")
print(test_df["isFraud"].value_counts())
print(test_df["isFraud"].value_counts(normalize=True) * 100)

TRAIN
isFraud
0    6232481
1       6559
Name: count, dtype: int64
isFraud
0    99.894872
1     0.105128
Name: proportion, dtype: float64

TEST
isFraud
0    121926
1      1654
Name: count, dtype: int64
isFraud
0    98.661596
1     1.338404
Name: proportion, dtype: float64


In [17]:
print(
    "Training fraud cases:",
    train_df["isFraud"].sum()
)

print(
    "Testing fraud cases:",
    test_df["isFraud"].sum()
)

Training fraud cases: 6559
Testing fraud cases: 1654


In [19]:
# speerating x and y for training and testing
X_train = train_df.drop(columns=["isFraud"])
y_train = train_df["isFraud"]

X_test = test_df.drop(columns=["isFraud"])
y_test = test_df["isFraud"]

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(6239040, 17) (6239040,)
(123580, 17) (123580,)


In [20]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score,
    roc_auc_score
)

dummy = DummyClassifier(strategy="most_frequent")

dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_test)

In [21]:
print(classification_report(
    y_test,
    y_pred_dummy,
    digits=4
))

              precision    recall  f1-score   support

           0     0.9866    1.0000    0.9933    121926
           1     0.0000    0.0000    0.0000      1654

    accuracy                         0.9866    123580
   macro avg     0.4933    0.5000    0.4966    123580
weighted avg     0.9734    0.9866    0.9800    123580



/Users/maryamellathy/Desktop/FinGaurd/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maryamellathy/Desktop/FinGaurd/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maryamellathy/Desktop/FinGaurd/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [22]:
confusion_matrix(
    y_test,
    y_pred_dummy
)

array([[121926,      0],
       [  1654,      0]])

In [23]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    y_test,
    y_pred_dummy
)

print(f"Accuracy: {accuracy:.6f}")

Accuracy: 0.986616


In [24]:
#fraud recall 
cm = confusion_matrix(y_test, y_pred_dummy)

tn, fp, fn, tp = cm.ravel()

print("True negatives:", tn)
print("False positives:", fp)
print("False negatives:", fn)
print("True positives:", tp)

fraud_recall = tp / (tp + fn)

print(f"\nFraud recall: {fraud_recall:.4f}")

True negatives: 121926
False positives: 0
False negatives: 1654
True positives: 0

Fraud recall: 0.0000
